# Week 08 - Keypoint Features and Matching

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Explain what makes a good **keypoint** (corner) and why corners are repeatable.
- Detect keypoints with **Harris**, **Shi-Tomasi**, **SIFT** and **ORB**.
- Describe and match **feature descriptors** using the **ratio test**.
- Estimate a **homography** with RANSAC and use it for panorama stitching.

### Where this fits
So far we segmented regions by colour and intensity. Recognition needs *distinctive* points that survive rotation, scale and illumination changes. These are **keypoints** plus their **descriptors**.

## 1. Setup

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install opencv-python matplotlib numpy ipywidgets

In [ ]:
import sys
sys.path.append("resources/scripts")
import cv2, numpy as np
from cvhelpers import show, concept_map, image_grid

print("OpenCV:", cv2.__version__)

## 2. The keypoint pipeline at a glance

In [ ]:
concept_map([
    "Image pair",
    "Detect keypoints (Harris / SIFT / ORB)",
    "Compute descriptors (128-D / 256-bit)",
    "Match descriptors (BF or FLANN) + ratio test",
    "Robust geometric fit (RANSAC homography)",
    "Application: stitching, tracking, 3D reconstruction"
], title="Feature-based recognition pipeline")

## 3. Guided example - Harris corner detector
A corner is where intensity changes **strongly in two directions**. Harris computes the structure tensor and scores each pixel with
$$R = \det(M) - k\,(\mathrm{trace}(M))^2.$$
Large positive $R$ = corner.

In [ ]:
img = cv2.imread("resources/images/hasan.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

grayf = np.float32(gray)
R = cv2.cornerHarris(grayf, blockSize=2, ksize=3, k=0.04)
R = cv2.dilate(R, None)

vis = img.copy()
vis[R > 0.01 * R.max()] = [0, 0, 255]
show(img, vis, titles=["Original", "Harris corners (red)"])

###  Interactive exploration - Harris threshold
Move the slider. A **low** threshold flags texture and noise; a **high** threshold keeps only strong corners.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def harris_demo(threshold_percent=1):
    v = img.copy()
    v[R > (threshold_percent / 100.0) * R.max()] = [0, 0, 255]
    show(v, titles=[f"threshold = {threshold_percent}% of max"], figsize=(6, 6))

interact(harris_demo, threshold_percent=widgets.IntSlider(min=1, max=50, step=1, value=1))

## 4. Guided example - Shi-Tomasi (goodFeaturesToTrack)
Shi-Tomasi uses the **smaller eigenvalue** as the score and is the basis of `goodFeaturesToTrack`. It is fast and gives well-spread corners.

In [ ]:
corners = cv2.goodFeaturesToTrack(gray, maxCorners=100, qualityLevel=0.01, minDistance=10)
vis = img.copy()
for c in corners.astype(int):
    x, y = c.ravel()
    cv2.circle(vis, (x, y), 4, (0, 255, 0), -1)
show(vis, titles=[f"Shi-Tomasi: {len(corners)} corners"], figsize=(6, 6))

## 5. Guided example - SIFT descriptors
SIFT detects **scale-space** extrema and assigns a **128-dimensional** descriptor that is invariant to scale and (mostly) rotation. It is accurate but slower.

In [ ]:
sift = cv2.SIFT_create(nfeatures=500)
kp_sift, des_sift = sift.detectAndCompute(gray, None)
vis = cv2.drawKeypoints(img, kp_sift, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
print("SIFT keypoints:", len(kp_sift), "| descriptor shape:", None if des_sift is None else des_sift.shape)
show(vis, titles=["SIFT keypoints (size = scale, line = orientation)"], figsize=(6, 6))

## 6. Guided example - ORB (fast alternative)
ORB = **O**riented **FAST** + **r**otated **BRIEF**. It is binary, very fast, and free from patent issues - ideal for real-time and embedded systems.

In [ ]:
orb = cv2.ORB_create(nfeatures=500)
kp_orb, des_orb = orb.detectAndCompute(gray, None)
vis = cv2.drawKeypoints(img, kp_orb, None, color=(255, 0, 0))
print("ORB keypoints:", len(kp_orb), "| descriptor shape:", None if des_orb is None else des_orb.shape)
show(vis, titles=["ORB keypoints"], figsize=(6, 6))

| Property | SIFT | ORB |
|---|---|---|
| Descriptor type | float, 128-D | binary, 256-bit |
| Speed | slow | very fast |
| Rotation invariant | yes | yes |
| Scale invariant | yes | partially |
| Best for | accuracy, stitching | real-time, embedded |

## 7. Guided example - descriptors, matching and the ratio test
We make a rotated, scaled copy of the image, match descriptors with a **brute-force matcher**, and keep only *good* matches using Lowe's **ratio test**: a match is accepted only if the best match is clearly better than the second best.

In [ ]:
# Create a transformed copy to match against
M = cv2.getRotationMatrix2D((gray.shape[1] / 2, gray.shape[0] / 2), 20, 0.8)
warped = cv2.warpAffine(img, M, (gray.shape[1], gray.shape[0]))
gray_w = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)

kp1, des1 = sift.detectAndCompute(gray, None)
kp2, des2 = sift.detectAndCompute(gray_w, None)

bf = cv2.BFMatcher(cv2.NORM_L2)
matches = bf.knnMatch(des1, des2, k=2)  # two nearest neighbours

def ratio_filter(matches, ratio):
    return [m for m, n in matches if m.distance < ratio * n.distance]

good = ratio_filter(matches, 0.75)
print("Total matches:", len(matches), "| good after ratio test:", len(good))
draw = cv2.drawMatches(img, kp1, warped, kp2, good, None, flags=2)
show(draw, titles=["Good matches (ratio < 0.75)"], figsize=(12, 6))

###  Interactive exploration - effect of the ratio threshold
A **strict** ratio (e.g. 0.6) keeps fewer but more reliable matches. A **loose** ratio (e.g. 0.9) keeps more matches but adds outliers.

In [ ]:
def match_demo(ratio=0.75):
    g = ratio_filter(matches, ratio)
    d = cv2.drawMatches(img, kp1, warped, kp2, g, None, flags=2)
    show(d, titles=[f"ratio = {ratio:.2f}  ->  {len(g)} matches"], figsize=(12, 6))

interact(match_demo, ratio=widgets.FloatSlider(min=0.5, max=0.95, step=0.01, value=0.75))

## 8. Guided example - homography with RANSAC
A **homography** is a $3\times3$ matrix mapping one plane to another. Matches contain outliers, so we fit the homography with **RANSAC**, which finds a model supported by the most inliers.

In [ ]:
src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
inliers = int(mask.sum())
print("Homography matrix:\n", np.round(H, 3))
print("Inliers:", inliers, "out of", len(good))

# Verify: warp the original with H and compare to the transformed copy
h, w = gray.shape
aligned = cv2.warpPerspective(img, H, (w, h))
show(warped, aligned, titles=["Reference (warped)", "Aligned via estimated H"], figsize=(12, 6))

## 9. Exercise (complete the code)

1. Replace SIFT with ORB in the matching pipeline (`cv2.BFMatcher(cv2.NORM_HAMMING)`).
2. Compare the number of good matches and the runtime of SIFT vs ORB using `time.perf_counter()`.
3. Which would you deploy on a drone for real-time landing-marker detection?

In [ ]:
# TODO: ORB matching + timing comparison


## 10. Challenge (independent)

Build a **panorama stitcher** for two overlapping photos (you can create the second by warping the first). Estimate the homography from ORB matches, then use `cv2.warpPerspective` to place both images on a common canvas and blend them with `np.maximum` or feathering.

In [ ]:
# Your code here


## 11. Check your understanding (Q&A)

<details><summary><b>Q1. Why is a corner more useful than a flat region or an edge?</b></summary>

A flat region looks the same in every direction, and an edge looks the same along its direction. A corner changes in two directions, so its location can be determined precisely and matched reliably.
</details>

<details><summary><b>Q2. What does the ratio test actually test?</b></summary>

It checks that the nearest descriptor is significantly closer than the second nearest. If the two are similar, the match is ambiguous and likely an outlier.
</details>

<details><summary><b>Q3. Why do we still need RANSAC if the ratio test already removes many outliers?</b></summary>

The ratio test cannot remove all outliers. RANSAC fits the geometric model to the largest consistent set of matches, so a few wrong matches do not corrupt the homography.
</details>

<details><summary><b>Q4. Give two engineering uses of homography.</b></summary>

Document scanning / dewarping, and panorama stitching. Also planar pose estimation for robot grasping and augmented reality markers.
</details>

## 12. Further reading & self-exploration
- OpenCV feature detection and description: https://docs.opencv.org/4.x/db/d27/tutorial_py_table_of_contents_feature2d.html
- OpenCV feature matching and homography: https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html
- SIFT paper: Lowe, D. (2004), *Distinctive Image Features from Scale-Invariant Keypoints*.
- ORB paper: Rublee et al. (2011), *ORB: An efficient alternative to SIFT or SURF*.
- Wikipedia - Scale-invariant feature transform: https://en.wikipedia.org/wiki/Scale-invariant_feature_transform
- PyImageSearch - feature matching tutorials: https://pyimagesearch.com/
- scikit-image feature module: https://scikit-image.org/docs/stable/api/skimage.feature.html

**Try next:** stitch two photos from your phone, or detect a planar marker for a simple AR overlay.

## 13. Key takeaways
- Keypoints = distinctive corners; descriptors = numerical signatures.
- **SIFT** = accurate, float, slower. **ORB** = fast, binary, embedded-friendly.
- Match with BF/FLANN, filter with the **ratio test**.
- Fit geometry robustly with **RANSAC homography** (4 DOF+).
- This pipeline underpins stitching, tracking, SLAM and 3D reconstruction.